In [1]:
# --- Configuration: Define Data Lake Paths ---
# Using the Bronze Layer as the raw data entry point in Azure Data Lake Storage (ADLS)
storage_account = "liufei2026storage"
container = "data"
file_path = f"abfss://{container}@{storage_account}.dfs.core.windows.net/01-bronze/olist_ecommerce/olist_orders_dataset.csv"

# --- Data Ingestion: PySpark Read Operations ---
# We enable 'inferSchema' for initial data type detection 
# and 'header' to treat the first row as column names.
df_orders = spark.read.format("csv") \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .load(file_path)

# --- Data Validation ---
# Quick inspection of the first 5 records to verify structure and loading
display(df_orders.limit(5))

StatementMeta(Sparkpool1, 3, 2, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 8a03ae72-965d-46a7-b645-fb1d589385d5)

In [2]:
from pyspark.sql.functions import col, to_timestamp

# --- Transformation: Standardizing Temporal Data ---
# Define list of columns requiring conversion from String to Timestamp for time-series analysis
timestamp_cols = [
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date"
]

# --- Iterative Type Casting (Bronze to Silver) ---
# Transitioning data to the Silver layer by enforcing strict timestamp formatting
df_silver_orders = df_orders
for c in timestamp_cols:
    df_silver_orders = df_silver_orders.withColumn(c, to_timestamp(col(c), "yyyy-MM-dd HH:mm:ss"))

# --- Schema Validation ---
# Verify that all target columns have been successfully cast to 'timestamp' type
df_silver_orders.printSchema()
display(df_silver_orders.limit(5))

StatementMeta(Sparkpool1, 3, 3, Finished, Available, Finished, False)

root
 |-- order_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- order_status: string (nullable = true)
 |-- order_purchase_timestamp: timestamp (nullable = true)
 |-- order_approved_at: timestamp (nullable = true)
 |-- order_delivered_carrier_date: timestamp (nullable = true)
 |-- order_delivered_customer_date: timestamp (nullable = true)
 |-- order_estimated_delivery_date: timestamp (nullable = true)



SynapseWidget(Synapse.DataFrame, 975a939f-db32-4e2c-9958-28f0d9073bf1)

In [3]:
# --- 1. Data Persistence (Parquet Format) ---
# Saving the cleaned orders to Silver layer using Parquet for optimized storage and query performance.
silver_path = f"abfss://{container}@{storage_account}.dfs.core.windows.net/02-silver/olist_ecommerce/orders"
df_silver_orders.write.mode("overwrite").parquet(silver_path)

# --- 2. Modular ETL Framework ---
# Defining a reusable processing function to standardize the Bronze-to-Silver transition.
# Features: Dynamic schema inference, automated timestamp casting, and custom transformation injection.
def process_silver_v2(file_name, target_folder, date_cols=[], custom_transform=None):
    # Base Ingestion
    path = f"abfss://{container}@{storage_account}.dfs.core.windows.net/01-bronze/olist_ecommerce/{file_name}.csv"
    df = spark.read.option("header", "true").option("inferSchema", "true").csv(path)
    
    # Generic Date Transformation
    for col_name in date_cols:
        df = df.withColumn(col_name, to_timestamp(col(col_name), "yyyy-MM-dd HH:mm:ss"))
    
    # Executing Business-Specific Logic (Dependency Injection)
    if custom_transform:
        df = custom_transform(df)
    
    # Parquet Persistence with Overwrite
    output_path = f"abfss://{container}@{storage_account}.dfs.core.windows.net/02-silver/olist_ecommerce/{target_folder}"
    df.write.mode("overwrite").parquet(output_path)
    print(f"✅ Table {target_folder} processed and saved to Silver.")

# --- 3. Business-Specific Transformation Rules ---
# Customer: Enforce Zip Code as String to preserve leading zeros
def transform_customers(df):
    return df.withColumn("customer_zip_code_prefix", col("customer_zip_code_prefix").cast("string"))

# Products: Handle missing category names for downstream analysis
def transform_products(df):
    return df.fillna("Unknown", subset=["product_category_name"])

# Geolocation: Deduplication based on unique zip codes for mapping accuracy
def tansform_geograph(df):
    return df.withColumn("geolocation_zip_code_prefix", col("geolocation_zip_code_prefix").cast("string")) \
             .dropDuplicates(["geolocation_zip_code_prefix"])

# --- 4. Main Execution Pipeline ---
# Batch processing of all Olist datasets using the defined framework
datasets = [
    ("olist_customers_dataset", "customers", [], transform_customers),
    ("olist_products_dataset", "products", [], transform_products),
    ("olist_geolocation_dataset", "geolocation", [], tansform_geograph),
    ("olist_order_items_dataset", "items", ["shipping_limit_date"], None),
    ("olist_order_payments_dataset", "payments", [], None),
    ("olist_order_reviews_dataset", "reviews", ["review_creation_date", "review_answer_timestamp"], None),
    # ... more datasets
]



StatementMeta(Sparkpool1, 3, 4, Finished, Available, Finished, False)